In [1]:
# This is timport datarobot as dr
import pandas as pd
import numpy as np
from datetime import datetime
from scipy.stats import mode
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
pd.set_option('display.max_rows', 3000)

# 1 - Ingest Data

In [2]:
# Modulo1632_REC21_2023 = pd.read_csv("C:/Users/linoc/OneDrive/Encoder/03_partos/01_raws/2024/968-Modulo1632/968-Modulo1632/REC21_2024.csv", low_memory=False)
# Modulo1632_REC21_2023.info(verbose = True, show_counts = True)


# Build base directory (go up one level from current /02_scripts)
base_dir = os.path.dirname(os.getcwd())   # → c:\Users\linoc\OneDrive\Encoder\03_partos

# Build the path to the target CSV
csv_path_Modulo1633_REC41_2024_fil = os.path.join(base_dir,"03_bases_intermediarios", "Modulo1633_REC41_2024_fil.csv")
csv_path_Modulo1633_REC94_2024_fil = os.path.join(base_dir,"03_bases_intermediarios", "Modulo1633_REC94_2024_fil.csv")
# Build the path to the target CSV
csv_path_Modulo1631_REC91_2024 = os.path.join(base_dir,"01_raws", "2024", "968-Modulo1631", "968-Modulo1631", "REC91_2024.csv")
csv_path_Modulo1632_RE223132_2024= os.path.join(base_dir,"01_raws", "2024", "968-Modulo1632", "968-Modulo1632", "RE223132_2024.csv")
csv_path_Modulo1635_RE516171_2024 = os.path.join(base_dir,"01_raws", "2024", "968-Modulo1635", "968-Modulo1635", "RE516171_2024.csv")

# Load the dataset
Modulo1633_REC41_2024_fil = pd.read_csv(csv_path_Modulo1633_REC41_2024_fil, low_memory=False)
Modulo1633_REC94_2024_fil = pd.read_csv(csv_path_Modulo1633_REC94_2024_fil, low_memory=False)
Modulo1631_REC91_2024 = pd.read_csv(csv_path_Modulo1631_REC91_2024, low_memory=False)
Modulo1632_RE223132_2024 = pd.read_csv(csv_path_Modulo1632_RE223132_2024, low_memory=False)
Modulo1635_RE516171_2024 = pd.read_csv(csv_path_Modulo1635_RE516171_2024, low_memory=False)

Modulo1633_REC41_2024_fil.shape, Modulo1633_REC94_2024_fil.shape, Modulo1631_REC91_2024.shape, Modulo1632_RE223132_2024.shape, Modulo1635_RE516171_2024.shape

((17608, 467), (17608, 269), (37117, 343), (34252, 149), (34252, 84))

In [3]:
Modulo1632_RE223132_2024.CASEID.nunique()

34252

# 2 Data Wrangling

# 2.1 selection variable type

In [4]:
def categorize_columns(df, key_variables):
    numeric_cols = []
    categorical_cols = []
    dummy_cols = []

    features = df.columns.to_list()
    #features.remove(target_variable)
    features =[item for item in features  if item not in key_variables]

    # Iterate over each column in DataFrame
    for col in features:
        if pd.api.types.is_numeric_dtype(df[col]):
            # Check if the column is a dummy variable
            unique_values = pd.Series(df[col].dropna().unique())
            
            if unique_values.isin([0, 1]).all() and unique_values.size <= 2:
                dummy_cols.append(col)
            else:
                numeric_cols.append(col)
        elif pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
            categorical_cols.append(col)
    
    print(len(numeric_cols)),print(len(categorical_cols)), print(len(dummy_cols)) 
    
    return numeric_cols, categorical_cols, dummy_cols

##  2.2 - Treatment of Numerical Data: Outlier, fill Missing Values and Normalize

In [5]:
def fill_missing_with_median_coding(df):
    """
    Drops values greater than the 99th percentile, fills missing values in each column 
    of the DataFrame with the median of that column, and applies MinMax scaling to each 
    numeric column.

    Parameters:
    df (pd.DataFrame): The DataFrame with missing values.

    Returns:
    pd.DataFrame: A DataFrame with values above 99th percentile dropped, missing values 
    filled, and numeric columns scaled from 0 to 1.
    """

    # Calculate the 99th percentile for each column and drop rows with any value above it
    
    for column in df.columns.to_list():
        percentile_99 = df[column].quantile(0.99)
        max_value_below_99 = df[df[column] < percentile_99][column].max()
        df[column] = df[column].apply(lambda x: max_value_below_99 if x > percentile_99 else x)
        
    df_filled = df.copy()
    
    for column in df_filled.columns:
        median_value = df_filled[column].median()
        df_filled[column].fillna(median_value, inplace=True)

        scaler = MinMaxScaler()
        df_filled[column] = scaler.fit_transform(df_filled[[column]]) 

    print(df_filled.shape)
    
    return df_filled

## 2.3 - Treatment of categorical  Data: Fill Missing Values and encoding

In [6]:
# function missing values 
def fill_missing_with_mode(df):
    """
    Fills missing values in each column of the DataFrame with the median of that column.

    Parameters:
    df (pd.DataFrame): The DataFrame with missing values.

    Returns:
    pd.DataFrame: A DataFrame with missing values filled with the median of their respective columns.
    """
    df_filled = df.copy()
    
    for column in df_filled.columns:
        mode_value = df_filled[column].mode()[0]
        #df_filled[column].fillna(median_value, inplace=True)
        df_filled[column] = df_filled[column].fillna(mode_value)
    
    return df_filled


# function encoding 
def target_encode(df, categorical_columns, target_column):
    # Create a copy of the DataFrame to avoid modifying the original data
    df_encoded = df.copy()
    
    # For each categorical feature, perform target encoding
    for column in categorical_columns:
        # Create a dictionary of category: average target
        target_means = df.groupby(column)[target_column].mean()
        # Map the categorical features to these target averages
        df_encoded[column] = df[column].map(target_means)
    
    df_encoded.drop(columns = target_column, inplace = True)
        
    return df_encoded

## 2.4 Flag

In [8]:
#df_load_dummy_cols =  fill_missing_with_mode(df_load_join[dummy_cols])

# 3 - DBS 

## 3.1 csv_path_Modulo1633_REC41_2024_fil

In [9]:
key_variables =['CASEID']
numeric_cols, categorical_cols, dummy_cols = categorize_columns (Modulo1633_REC41_2024_fil, key_variables)

143
0
323


In [10]:
df_numeric_cols = fill_missing_with_median_coding(Modulo1633_REC41_2024_fil[numeric_cols])

C:\Users\linoc\AppData\Local\Temp\ipykernel_62448\3211958872.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].apply(lambda x: max_value_below_99 if x > percentile_99 else x)
C:\Users\linoc\AppData\Local\Temp\ipykernel_62448\3211958872.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].apply(lambda x: max_value_below_99 if x > percentile_99 else x)
C:\Users\linoc\AppData\Local\Temp\ipykernel_62448\3211958872.py:20: SettingWithCopyWarning: 
A value is tryi

(17608, 143)


C:\Users\linoc\AppData\Local\Temp\ipykernel_62448\3211958872.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filled[column].fillna(median_value, inplace=True)
C:\Users\linoc\AppData\Local\Temp\ipykernel_62448\3211958872.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [ ]:
#df_categorical_cols = fill_missing_with_mode(Modulo1633_REC41_2024_fil[categorical_cols])
#df_categorical_cols = target_encode((pd.concat([df_categorical_cols, Modulo1633_REC41_2024_fil['SURVEY_TARGET_SSPIVISN40']], axis=1)), categorical_cols, 'SURVEY_TARGET_SSPIVISN40')

In [11]:
df_load_dummy_cols =  fill_missing_with_mode(Modulo1633_REC41_2024_fil[dummy_cols])

In [13]:
Modulo1632_RE223132_2024_clear = pd.concat([df_numeric_cols, df_load_dummy_cols, Modulo1633_REC41_2024_fil[key_variables]], axis=1)
Modulo1632_RE223132_2024_clear.info(verbose = True, show_counts = True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17608 entries, 0 to 17607
Data columns (total 467 columns):
 #    Column     Non-Null Count  Dtype  
---   ------     --------------  -----  
 0    M3A        17608 non-null  float64
 1    M3B        17608 non-null  float64
 2    M3C        17608 non-null  float64
 3    M3D        17608 non-null  float64
 4    M3G        17608 non-null  float64
 5    M3H        17608 non-null  float64
 6    M3K        17608 non-null  float64
 7    M3N        17608 non-null  float64
 8    M17        17608 non-null  float64
 9    M66        17608 non-null  float64
 10   ID1_2024   17608 non-null  float64
 11   M1_<NA>    17608 non-null  float64
 12   M1A_<NA>   17608 non-null  float64
 13   M1B_<NA>   17608 non-null  float64
 14   M1D_<NA>   17608 non-null  float64
 15   M5_0       17608 non-null  float64
 16   M5_1       17608 non-null  float64
 17   M5_10      17608 non-null  float64
 18   M5_11      17608 non-null  float64
 19   M5_12      17608 non-nu

# 3 Summary and save database

In [ ]:
df = pd.concat([df_load_dummy_cols, df_categorical_cols, df_numeric_cols, df_load_join[key_variables]], axis=1)
df.shape

(51837, 1984)

In [ ]:
df_load_dummy_cols.shape, df_categorical_cols.shape, df_numeric_cols.shape, df_load_join[key_variables].shape

((51837, 500), (51837, 56), (51837, 1422), (51837, 6))

In [ ]:
dr.Dataset.create_from_in_memory_data(
    data_frame=df,
    fname="02_HealthInsuranceDirectPay_20240910_DataWrangling.csv",
)

Dataset(name='02_HealthInsuranceDirectPay_20240910_DataWrangling.csv', id='66e24bffaa0b52f71b96a816')

In [ ]:
# fin 

In [ ]:
dataset_id = '66e24bffaa0b52f71b96a816'

# Initialize the DataRobot client
dr.Client(token=token, endpoint=endpoint)

# Fetch the dataset
dataset = Dataset.get(dataset_id)

# Convert the dataset to a DataFrame
df= dataset.get_as_dataframe(low_memory=False)
df.shape

(51837, 1984)

In [ ]:
df[['SURVEY_CONSMR_ID_SEED',
                    'SURVEY_KLNK_SEED',
                    'SURVEY_TARGET_SSPIVISN40',
                    'SURVEY_SURVEY_YEAR',
                    'AMER_CONSMR_ID',
                    'AMER_KLNK']].head(100)


,SURVEY_CONSMR_ID_SEED,SURVEY_KLNK_SEED,SURVEY_TARGET_SSPIVISN40,SURVEY_SURVEY_YEAR,AMER_CONSMR_ID,AMER_KLNK
0,1000002388,47815121910,0,2022,1000002388,47815121910
1,1000018830,8346837120006,0,2022,1000018830,8346837120006
2,1000019140,549562122010,0,2022,1000019140,549562122010
3,1000088487,7794120812,0,2022,1000088487,7794120812
4,1000108328,877907120006,0,2022,1000108328,877907120006
5,1000201182,6548773120006,0,2024,1000201182,6548773120006
6,1000220182,4490121681,0,2024,1000220182,4490121681
7,1000222402,83920170702,0,2022,1000222402,83920170702
8,1000228388,478916120904,0,2024,1000228388,478916120904
9,1000228998,5965842120006,0,2024,1000228998,5965842120006


In [ ]:
df_survey2024.SURVEY_TARGET_SSPIVISN40.value_counts()

0    27028
1     2357
Name: SURVEY_TARGET_SSPIVISN40, dtype: int64

In [ ]:
dr.Dataset.create_from_in_memory_data(
    data_frame=df_survey2024,
    fname="02_HealthInsuranceDirectPay_20240910_DataWrangling_survey2024.csv",
)

Dataset(name='02_HealthInsuranceDirectPay_20240910_DataWrangling_survey2024.csv', id='66e3615bbd17cb6cc21f9889')